# LiteLLM Model class
- Supports calling different model providers through the same API interface. See docs [here](https://docs.litellm.ai/docs/providers)
- Enables cost and token tracking
- Helper classes for APIModelConfig and APIStats. APIModelConfig is used to set model configuration parameters and APIStats is used to track cost and token counts.


In [1]:
import sys, subprocess, importlib.util, os

print("=== Python Executable ===")
print(sys.executable)

print("\n=== sys.path ===")
for p in sys.path:
    print(p)

print("\n=== Installed scikit-learn versions (if any) ===")
subprocess.run(["pip", "show", "scikit-learn"])

print("\n=== Checking if sklearn is importable ===")
spec = importlib.util.find_spec("sklearn")
print("Found sklearn:" if spec else "❌ sklearn not found")

print("\n=== Current Working Directory ===")
print(os.getcwd())


=== Python Executable ===
/data4/anika/UCSBCounterspeechProject/.venv/bin/python

=== sys.path ===
/data4/anika/UCSBCounterspeechProject
/home/anika/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python311.zip
/home/anika/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11
/home/anika/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11/lib-dynload

/data4/anika/UCSBCounterspeechProject/.venv/lib/python3.11/site-packages

=== Installed scikit-learn versions (if any) ===
Name: scikit-learn
Version: 1.7.2
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License-Expression: BSD-3-Clause
Location: /data4/anika/UCSBCounterspeechProject/.venv/lib/python3.11/site-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: 

=== Checking if sklearn is importable ===
Found sklearn:

=== Current Working Directory ===
/data4/anika/UCSBCounterspeechProject


In [2]:
!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install --upgrade pip setuptools wheel


Looking in links: /tmp/tmp36yqwcrr


In [3]:
!{sys.executable} -m pip install -U scikit-learn

In [4]:
# Imports

from __future__ import annotations

import logging
from logging import Logger
from typing import Any, Dict, Iterable, Iterator, List, Optional
import json, io
from pathlib import Path

# External libraries
from dotenv import load_dotenv
import litellm
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_not_exception_type
from pydantic import BaseModel, Field, field_validator
from litellm.types.utils import ModelResponse, Choices
from litellm.exceptions import (
    APIError,
    AuthenticationError,
    BadRequestError,
    ContextWindowExceededError,
    NotFoundError,
    PermissionDeniedError,
)

# Add sklearn for evaluation metrics
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report


In [5]:
# Helper Classes


# Define a MODEL_ALIASES dictionary to easily map long model names to short ones for cli usage. An example is given below.
MODEL_ALIASES = {
    "gpt4o": "gpt-4o",
    "bedrock-claude-3.5-sonnet": "bedrock/arn:aws:bedrock:us-west-2:288380904485:inference-profile/us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    # Add more as needed
}

class APIModelConfig(BaseModel):
    """Configuration for the API model."""

    model_name: str = Field(
        default="gpt4o",
        description="Full litellm model name of the model. See https://docs.litellm.ai/docs/providers",
    )
    temperature: float = Field(
        default=1.0,
        description="The temperature for the model. Default to 1.0 since OpenAI O series does not support other temperature values.",
    )
    top_p: float = Field(default=1.0, description="The top-p value for the model.")
    max_tokens: int = Field(default=1000, description="The maximum number of tokens to generate.")
    host_url: str | None = Field(
        default=None, description="The base URL for the model. Used for some custom model providers such as vLLM/ollama."
    )
    completion_kwargs: dict[str, Any] = Field(
        default={}, description="additional kwargs for the litellm.completion call."
    )

    @field_validator("model_name")
    @classmethod
    def validate_model_name(cls, v: str) -> str:
        """Validate the model name.

        Args:
            v: The model name to validate.

        Returns:
            The validated model name.
        """
        if v in MODEL_ALIASES:
            return MODEL_ALIASES[v]
        return v


# destination: src/models.py
class APIStats(BaseModel):
    """Statistics for the API model."""

    cost: float = 0.0
    tokens_received: int = 0
    tokens_sent: int = 0
    api_calls: int = 0

    def __add__(self, other: APIStats) -> APIStats:
        """Add two APIStats objects together.

        Args:
            other: The other APIStats object to add to the current object.

        Returns:
            A new APIStats object with the sum of the two objects.

        Raises:
            TypeError: If other is not an APIStats object.
        """
        if not isinstance(other, APIStats):
            raise TypeError("APIStats objects can only be added to other APIStats objects")

        return APIStats(**{field: getattr(self, field) + getattr(other, field) for field in self.model_fields})


In [6]:
class LiteLLMModel:
    """A wrapper for the LiteLLM API. Inherit this class to make it a stateful model.

    Cost state management is included in this class, so the inherited classes can implement their own cost management or reuse this class.
    """

    def __init__(self, config: APIModelConfig, name: str, logger: logging.Logger) -> None:
        """Initialize the LiteLLM API Model.

        Args:
            config: The configuration for the LiteLLM API Model
            name: The name of the LiteLLM API Model
            logger: The logger for the LiteLLM API Model
        """
        self.name = name
        self.config = config
        self.stats = APIStats()
        self.logger = logger
        self._setup_client()

    def _setup_client(self) -> None:
        self.model_name = self.config.model_name
        self.model_max_input_tokens = litellm.model_cost.get(self.model_name, {}).get("max_input_tokens")
        self.model_max_output_tokens = litellm.model_cost.get(self.model_name, {}).get("max_output_tokens")
        self.lm_provider = litellm.model_cost.get(self.model_name, {}).get("litellm_provider")
        if self.lm_provider is None or self.config.host_url is not None:
            self.logger.warning(
                f"Using custom host URL: {self.config.host_url}. Cost management will not be available. Register the model with litellm to enable cost management. See https://docs.litellm.ai/docs/completion/token_usage#9-register_model"
            )

    def reset_stats(self, other: APIStats) -> None:
        """Reset or replace the current API Statistics.

        Args:
            other: The other APIStats object to replace the current object with.
        """
        self.stats = other

    def update_stats(self, input_tokens: int, output_tokens: int, cost: float = 0.0) -> None:
        """Update API statistics with new usage information.

        Args:
            input_tokens (int): Number of tokens in the prompt
            output_tokens (int): Number of tokens in the response
            cost (float): Cost of the API call. Defaults to 0.0

        Returns:
            float: The calculated cost of the API call
        """
        self.stats.tokens_sent += input_tokens
        self.stats.tokens_received += output_tokens
        self.stats.cost += cost
        self.stats.api_calls += 1

    @retry(
        wait=wait_random_exponential(min=180, max=360),
        reraise=True,
        stop=stop_after_attempt(3),
        retry=retry_if_not_exception_type(
            (
                RuntimeError,
                NotFoundError,
                PermissionDeniedError,
                ContextWindowExceededError,
                APIError,
                AuthenticationError,
                BadRequestError,
            )
        ),
    )
    def query(
        self,
        messages: list[dict[str, Any]],
        tools: Optional[list[dict[str, Any]]] = None,
    ) -> litellm.types.utils.ModelResponse:
        """Query the model with messages and optional tools.

        Args:
            messages: List of messages in OpenAI format
            tools: Optional list of tool definitions in OpenAI format

        Returns:
            litellm.types.utils.ModelResponse: The complete response from the model
        """
        input_tokens = litellm.utils.token_counter(messages=messages, model=self.model_name)
        extra_args = {}
        if self.config.host_url:
            extra_args["api_base"] = self.config.host_url

        completion_kwargs = self.config.completion_kwargs.copy()
        if self.lm_provider == "anthropic":
            completion_kwargs["max_tokens"] = self.model_max_output_tokens

        # Add tools if provided
        if tools is not None:
            completion_kwargs["tools"] = tools

        try:
            response: litellm.types.utils.ModelResponse = litellm.completion(
                model=self.model_name,
                messages=messages,
                temperature=self.config.temperature,
                top_p=self.config.top_p,
                stream=False,
                drop_params=True,  # Automatically drop unsupported OpenAI params
                **completion_kwargs,
                **extra_args,
            )
        except Exception:
            self.logger.exception(f"Error querying {self.model_name} with tools")
            raise

        choices: litellm.types.utils.Choices = response.choices  # type: ignore
        output_content = choices[0].message.content or ""

        self.logger.debug(f"Input:\n{messages[-1]['content'] if messages else 'No messages'}")
        self.logger.debug(f"Response: {output_content}")

        # Calculate output tokens
        # For tool calls, we need to count tokens in the entire response
        if hasattr(choices[0].message, "tool_calls") and choices[0].message.tool_calls:
            # Include tool calls in token count
            tool_calls_text = str(choices[0].message.tool_calls)
            output_tokens = litellm.utils.token_counter(
                text=f"{output_content}{tool_calls_text}", model=self.model_name
            )
        else:
            output_tokens = litellm.utils.token_counter(text=output_content, model=self.model_name)

        # Update stats
        cost = litellm.cost_calculator.completion_cost(response)
        self.update_stats(input_tokens, output_tokens, cost)

        self.logger.debug(
            f"input_tokens={input_tokens:,}, output_tokens={output_tokens:,}, instance_cost={cost:.2f}, "
            f"total_tokens_sent={self.stats.tokens_sent:,}, total_tokens_received={self.stats.tokens_received:,}, total_cost={self.stats.cost:.2f}, total_api_calls={self.stats.api_calls:,}"
        )

        return response


In [7]:
# === SimplifiedTweets Parser ===
from __future__ import annotations
from pathlib import Path
import json
from typing import List, Dict, Any, Iterable, Optional

class SimplifiedTweetsParser:
    """
    Parser for the attached 50_examples_converted.json format.

    Expected keys per item:
      - ID (int)
      - Username (str)
      - Date (str, e.g. '2021-05-15 02:23:53+00:00')
      - Text (str)
      - Parent_Text (str | None)
      - Media_Photos (List[str])

    Methods:
      - build_user_content(...): returns a single Markdown string (safe for text-only models).
      - build_user_content_parts(...): returns OpenAI-style multimodal parts if you later swap to a vision model.
    """

    def __init__(self, json_path: str | Path) -> None:
        self.path = Path(json_path)
        if not self.path.exists():
            raise FileNotFoundError(f"Dataset not found: {self.path}")
        self._data: Optional[List[Dict[str, Any]]] = None

    def load(self) -> List[Dict[str, Any]]:
        if self._data is None:
            with self.path.open("r", encoding="utf-8") as f:
                self._data = json.load(f)
            if not isinstance(self._data, list):
                raise ValueError("Expected a JSON array of records.")
        return self._data

    def _norm_item(self, item: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "id": item.get("ID"),
            "username": item.get("Username"),
            "date": item.get("Date"),
            "text": item.get("Text") or "",
            "parent_text": item.get("Parent_Text"),
            "images": [u for u in (item.get("Media_Photos") or []) if isinstance(u, str) and u.strip()],
        }

    def iter_items(self) -> Iterable[Dict[str, Any]]:
        for raw in self.load():
            yield self._norm_item(raw)

    @staticmethod
    def _sanitize(s: str, max_len: Optional[int] = None) -> str:
        if s is None:
            return ""
        # keep newlines for readability; strip control chars
        s = s.replace("\r", "")
        if max_len is not None and len(s) > max_len:
            s = s[:max_len - 1] + "…"
        return s

    def build_user_content(
        self,
        *,
        include_parent: bool = True,
        include_images: bool = True,
        max_items: Optional[int] = 200,
        max_total_chars: int = 28000,
        text_truncate: Optional[int] = None,
    ) -> str:
        """
        Returns a single Markdown string suitable for:
          messages = [
            {"role": "system", "content": "..."},
            {"role": "user", "content": user_content},  # <— this field only
          ]

        Safe for text-only models. Image URLs are included inline.
        """
        items = list(self.iter_items())
        total = len(items)

        if max_items is not None:
            items = items[:max_items]

        header = [
            "# Dataset: 50_examples_converted.json",
            f"- Records in file: {total}",
            f"- Records included below: {len(items)}",
            f"- Fields: ID, Username, Date, Text, Parent_Text, Media_Photos",
            "",
            "## Tweets",
        ]

        body_lines: List[str] = []
        char_budget = max_total_chars - sum(len(h) + 1 for h in header)
        for idx, it in enumerate(items, start=1):
            id_ = it["id"]
            username = it["username"]
            date = it["date"]
            text = self._sanitize(it["text"], max_len=text_truncate)
            parent = self._sanitize(it["parent_text"], max_len=text_truncate)
            images = it["images"]

            block_lines = [
                f"### [{idx}] ID={id_}",
                f"- Username: @{username}",
                f"- Date: {date}",
                f"- Text: {text}" if text else "- Text: ",
            ]
            if include_parent:
                block_lines.append(f"- Parent_Text: {parent}" if parent else "- Parent_Text: ")
            if include_images:
                if images:
                    block_lines.append("- Images:")
                    for u in images:
                        # include as both markdown link and image tag for clarity
                        block_lines.append(f"  - {u}")
                        block_lines.append(f"  - ![]({u})")
                else:
                    block_lines.append("- Images: (none)")

            block = "\n".join(block_lines) + "\n"
            if char_budget - len(block) < 0:
                body_lines.append("\n[...truncated to respect max_total_chars...]\n")
                break
            body_lines.append(block)
            char_budget -= len(block)

        return "\n".join(header + body_lines)

    def build_user_content_parts(
        self,
        *,
        include_parent: bool = True,
        max_items: Optional[int] = 50,
        text_truncate: Optional[int] = None,
    ) -> List[Dict[str, Any]]:
        """
        Returns OpenAI-style multimodal content parts:
          [{"type":"text",...}, {"type":"image_url","image_url":{"url": ...}}, ...]

        Use ONLY if your model/endpoint supports image inputs.
        """
        items = list(self.iter_items())
        if max_items is not None:
            items = items[:max_items]

        parts: List[Dict[str, Any]] = [
            {
                "type": "text",
                "text": (
                    "Dataset: 50_examples_converted.json\n"
                    "Each item lists ID, Username, Date, Text, optional Parent_Text, and images.\n"
                    "Below are items with images attached as separate parts.\n"
                ),
            }
        ]
        for idx, it in enumerate(items, start=1):
            text = self._sanitize(it["text"], max_len=text_truncate)
            parent = self._sanitize(it["parent_text"], max_len=text_truncate)
            preface = f"[{idx}] ID={it['id']} | @{it['username']} | {it['date']}\nText: {text}"
            if include_parent:
                preface += f"\nParent_Text: {parent or ''}"
            parts.append({"type": "text", "text": preface})
            for u in it["images"]:
                parts.append({"type": "image_url", "image_url": {"url": u}})
        return parts

# Instantiate and build the content
parser = SimplifiedTweetsParser("50_examples_converted.json")
user_content = parser.build_user_content(
    include_parent=True,
    include_images=True,
    max_items=200,          # adjust as needed
    max_total_chars=28000,  # keep prompts reasonably sized
    text_truncate=None      # or e.g. 500 to clip very long texts
)

# If you later switch to a vision-capable model, you can instead do:
# user_content_parts = parser.build_user_content_parts(include_parent=True, max_items=50)

# Usage
- Create a new file called `.env` with your API key. **VERY IMPORTANT: Make sure this file is in your project's .gitignore**.
- Create a APIModelConfig instance with your model parameters.
- Create a LiteLLMModel instance with APIModelConfig and call the query method.

# Hate Speech Classification

In [8]:
# Add the openai API key to the .env file
OPENAI_API_KEY="<api_key>"


# In your script, import the APIModelConfig and LiteLLMModel classes
config = APIModelConfig(model_name="gpt-5-mini", temperature=1, top_p=1.0, max_tokens=1000)
model = LiteLLMModel(config=config, name="gpt-5-mini", logger=logging.getLogger(__name__))

# Prepare the messages
messages = [
    {"role": "system", "content": """You will classify individual texts (posts or replies) for hate speech and counterspeech. Each text is paired with a unique Twitter/X ID.
Your job: return the labels for each ID using the instructions detailed in the following two parts. Do not repeat the original text.

Part 1 – Hate Speech Classification
Fields to Add

1. hate_speech:
   * 1 if the text contains hate speech, 0 otherwise.
   * Hate speech = any statement that attacks, demeans, or incites harm against a person or group based on one or more protected characteristics (race, ethnicity, religion, nationality, gender, sexual orientation, disability, etc.).
   * Also mark as hate speech if the text endorses, supports, or agrees with a hateful statement.
     * For example, if a post says 'facts' or 'so true' in response to 'women are hella dumb', it should be labeled as hate speech because it expresses agreement with a hateful sentiment.
   * If the meaning or intent of a text is vague, ambiguous, or uncertain, **do not label it as hate speech**. Be conservative by default.

2. hate_speech_type:
   * 'explicit' → overt, direct use of slurs, severe derogatory language, or clear calls for violence.
     * Examples:
       * 'Gas all the Jews — it’s the only solution.' (call for genocide)
       * 'Kill every single one of those goat-fucking refugees, they don’t deserve to live.' (incitement to mass violence)
       * 'Lynch that nigger before he infects the neighborhood.' (explicit racial violence)
       * 'Fags aren’t even human. they should be wiped out.' (dehumanization + extermination)
   * 'implicit' → indirect, coded, sarcastic, or requiring context to detect — often euphemisms, plausible deniability, or dog whistles.
     * Examples:
       * 'What do you say to a black guy in a three piece suit? Will the defendant please rise.'
       * 'What part of a vegetable can’t you eat? The wheelchair.'
       * 'You want to hear a joke? Women’s rights.'
       * 'Why should you never marry a Jewish woman? When dinner time comes, they’re too scared to get near the oven.'
       * 'What’s the only positive thing about Ethiopia? The HIV tests.'

3. hate_speech_severity:
   * 'low' → Stereotypes, generalizations, or mildly demeaning remarks with low likelihood of direct harm.
     * Examples:
       * 'Women are too emotional to lead.'
       * 'Old people can’t understand technology.'
   * 'medium' → Targeted insults, harassment, or demeaning language likely to reinforce discrimination but without direct calls for violence.
     * Examples:
       * 'Send all the immigrants home, they don't belong here.'
       * 'Gays are disgusting and ruin the moral fabric of society.'
   * 'high' → Explicit calls for violence, genocide, or severe harassment likely to incite immediate harm.
     * Examples:
       * 'Burn every mosque to the ground.'
       * 'Every day when I see those fucking pride parades I want to beat up those faggots until I knock some sense into them.'
       * 'ENOUGH IS ENOUGH. We NEED to rid this country of those godforsaken Mexicans. We NEED to do whatever it takes.'

4. identity_targeted:
   * If hate_speech = 1, specify which identity group(s) are targeted. Choose from the following list (null if none):
     * 'women'
     * 'men'
     * 'Black'
     * 'White'
     * 'Jewish'
     * 'Muslim'
     * 'Christian'
     * 'Asian'
     * 'Latino/Hispanic'
     * 'LGBTQ+'
     * 'immigrants/refugees'
     * 'disabled people'
     * 'Indigenous'
     * 'other/unspecified'

Part 2 – Counterspeech Classification
Fields to Add

1. counterspeech:
   * 1 if the text directly addresses hate speech with the aim of reducing its impact, 0 otherwise.
   * To be labeled as counterspeech, the **parent text** (the post being replied to) **must be classified as hate speech**. If the parent is not hate speech, the text cannot be considered counterspeech.
   * If the intent or target of the response is vague, ambiguous, or uncertain, **do not label it as counterspeech**. Be conservative by default.

2. counterspeech_type:
   * If counterspeech = 1, choose the relevant categories out of the following (else null).

Category | Definition | Example
'presenting facts' | Correcting false or misleading statements with evidence. | 'Actually, refugee crime rates are lower than native-born citizens.'
'pointing out hypocrisy or contradictions' | Highlighting inconsistencies in the hate speaker's logic. | 'You preach family values but cheat on your spouse — maybe rethink judging others.'
'warning of consequences' | Pointing out possible legal, social, or personal repercussions. | 'Inciting violence online can get you charged with a felony.'
'affiliation' | Establishing shared identity or solidarity with the target or both groups. | 'I'm a veteran, and I fought for everyone's freedom — including Muslims.'
'denouncing hate speech' | Explicitly condemning the hateful message. | 'This is blatant racism and it's disgusting.'
'humor' | Using jokes, irony, or sarcasm to undermine hate. | 'Wow, you've unlocked the 'Medieval Thinking' achievement badge.'
'empathy/positive tone' | Expressing compassion, kindness, or unity. | 'We're all trying to live decent lives — hate helps no one.'
'hostile language' | Responding to hate with insults or profanity while countering the message. | 'Only an idiot would believe that garbage.'

3. dominant_counterspeech_type:
   * If there is no counterspeech present, put null.
   * If there is one type of counterspeech present, put that type.
   * If there are multiple types present, choose the most salient and emphasized form.

Output Format:
Return your response as a valid JSON array containing objects for each ID. Each object must have these exact fields:

'id': (the ID number)
'hate_speech': (0 or 1)
'hate_speech_type': ('explicit', 'implicit', or null)
'hate_speech_severity': ('low', 'medium', 'high', or null)
'identity_targeted': (group name from list above or null)
'counterspeech': (0 or 1)
'counterspeech_type': (one of the types listed above or null)
'dominant_counterspeech_type': (same as counterspeech_type if only one, or the dominant one if multiple, or null)

Return ONLY the JSON array, no other text, no markdown formatting."""
}
]
# Few-shot examples — replace or expand these
few_shot_examples = [
    {
        "user": """{
"ID": 1,
"Username": "example_user1",
"Date": "2021-01-01",
"Text": "All immigrants should go back to their countries.",
"Parent_Text": null,
"Media_Photos": []
}""",
        "assistant": """[
{"id": 1, "hate_speech": 1, "hate_speech_type": "explicit", "hate_speech_severity": "medium",
"counterspeech": 0, "counterspeech_type": null, "dominant_counterspeech_type": null}
]"""
    },
    {
        "user": """{
"ID": 2,
"Username": "example_user2",
"Date": "2021-01-02",
"Text": "That's disgusting. Everyone deserves respect.",
"Parent_Text": "All immigrants should go back to their countries.",
"Media_Photos": []
}""",
        "assistant": """[
{"id": 2, "hate_speech": 0, "hate_speech_type": null, "hate_speech_severity": null,
"counterspeech": 1, "counterspeech_type": "denouncing hate speech", "dominant_counterspeech_type": "denouncing hate speech"}
]"""
    }
]

# Add few-shot examples to the prompt
for ex in few_shot_examples:
    messages.append({"role": "user", "content": ex["user"]})
    messages.append({"role": "assistant", "content": ex["assistant"]})

# Then include your actual dataset text
messages.append({"role": "user", "content": user_content})

# Call the model
response = model.query(messages=messages)

for i, choice in enumerate(response.choices, 1):
    print(f"\n=== Choice {i} ===\n")
    print(choice.message.content)



=== Choice 1 ===

[
{"id": 1239062934799536128, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "medium", "identity_targeted": "Asian", "counterspeech": 0, "counterspeech_type": null, "dominant_counterspeech_type": null},
{"id": 1239063402569322496, "hate_speech": 1, "hate_speech_type": "explicit", "hate_speech_severity": "medium", "identity_targeted": "Asian", "counterspeech": 0, "counterspeech_type": null, "dominant_counterspeech_type": null},
{"id": 1542205641787334656, "hate_speech": 0, "hate_speech_type": null, "hate_speech_severity": null, "identity_targeted": null, "counterspeech": 0, "counterspeech_type": null, "dominant_counterspeech_type": null},
{"id": 1291456770293071872, "hate_speech": 1, "hate_speech_type": "explicit", "hate_speech_severity": "medium", "identity_targeted": "men", "counterspeech": 0, "counterspeech_type": null, "dominant_counterspeech_type": null},
{"id": 1466827328202412041, "hate_speech": 0, "hate_speech_type": null, "hate_spee

In [9]:
# === Batch Size + Ground Truth Evaluation (10 iterations per batch) ===
import time, json, numpy as np
from statistics import mean
from sklearn.metrics import accuracy_score, f1_score

# --- Load ground truth labels ---
with open("50_annotated_examples.json", "r", encoding="utf-8") as f:
    ground_truth = {int(d["id"]): d for d in json.load(f)}

# --- Prepare data ---
examples = list(parser.iter_items())
#batch_sizes = [1, 2, 4, 8, 16, 32, 50]
#num_iterations = 10
batch_sizes = [50]
num_iterations = 1
results_summary = []

# ==============================================================
# Helper functions
# ==============================================================

def build_messages(batch):
    """Constructs per-batch messages for model query."""
    return [
        messages[0],         # system prompt
        *messages[1:-1],     # few-shot examples
        {
            "role": "user",
            "content": "\n\n".join(
                [f"ID: {ex['id']}\nText: {ex['text']}\nParent_Text: {ex['parent_text']}" for ex in batch]
            ),
        },
    ]

def parse_model_output(raw_text):
    """Safely parse model JSON output into Python list."""
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        try:
            start = raw_text.find("[")
            end = raw_text.rfind("]") + 1
            return json.loads(raw_text[start:end])
        except Exception:
            return []

# ==============================================================
# Metric computation
# ==============================================================

def compute_metrics(preds, gold):
    """Compute metrics for binary, multiclass, and multilabel fields."""

    # --- Helper normalizers ---
    def safe_label(val):
        if not val or str(val).lower() in {"unknown", "none", "null", "nan"}:
            return "none"
        return str(val).strip()

    def normalize_identities(x):
        if x is None:
            return []
        if isinstance(x, str):
            return [x]
        if isinstance(x, list):
            return x
        return []

    # --- Binary fields ---
    y_true_hate, y_pred_hate = [], []
    y_true_counter, y_pred_counter = [], []

    # --- Multiclass fields ---
    y_true_type, y_pred_type = [], []
    y_true_severity, y_pred_severity = [], []
    y_true_counter_type, y_pred_counter_type = [], []

    # --- Multilabel (identities) ---
    base_identities = [
        'women', 'men', 'Black', 'White', 'Jewish', 'Muslim', 'Christian',
        'Asian', 'Latino/Hispanic', 'LGBTQ+', 'immigrants/refugees',
        'disabled people', 'Indigenous', 'other/unspecified'
    ]
    all_identities = set(base_identities)

    # Dynamically extend identity list with unseen ones
    for d in gold.values():
        for x in normalize_identities(d.get("identity_targeted")):
            all_identities.add(x)
    for p in preds.values():
        for x in normalize_identities(p.get("identity_targeted")):
            all_identities.add(x)
    all_identities = sorted(all_identities)

    y_true_id, y_pred_id = [], []

    # --- Collect gold and predicted values ---
    for pid, p in preds.items():
        g = gold.get(pid)
        if not g:
            continue

        # Binary
        y_true_hate.append(g["hate_speech"])
        y_pred_hate.append(p.get("hate_speech", 0))
        y_true_counter.append(g["counterspeech"])
        y_pred_counter.append(p.get("counterspeech", 0))

        # Multiclass (cleaned)
        y_true_type.append(safe_label(g.get("hate_speech_type")))
        y_pred_type.append(safe_label(p.get("hate_speech_type")))
        y_true_severity.append(safe_label(g.get("hate_speech_severity")))
        y_pred_severity.append(safe_label(p.get("hate_speech_severity")))
        y_true_counter_type.append(safe_label(g.get("counterspeech_type")))
        y_pred_counter_type.append(safe_label(p.get("counterspeech_type")))

        # Multilabel (normalized)
        gt_raw = normalize_identities(g.get("identity_targeted"))
        pr_raw = normalize_identities(p.get("identity_targeted"))
        gt_set, pr_set = set(gt_raw), set(pr_raw)
        y_true_id.append([1 if i in gt_set else 0 for i in all_identities])
        y_pred_id.append([1 if i in pr_set else 0 for i in all_identities])

    # --- Compute metrics safely ---
    metrics = {
        # Binary
        "hate_acc": round(accuracy_score(y_true_hate, y_pred_hate), 3),
        "hate_f1": round(f1_score(y_true_hate, y_pred_hate), 3),
        "counter_acc": round(accuracy_score(y_true_counter, y_pred_counter), 3),
        "counter_f1": round(f1_score(y_true_counter, y_pred_counter), 3),

        # Multiclass (macro F1 averages over all categories)
        "type_acc": round(accuracy_score(y_true_type, y_pred_type), 3),
        "type_f1": round(f1_score(y_true_type, y_pred_type, average="macro"), 3),

        "severity_acc": round(accuracy_score(y_true_severity, y_pred_severity), 3),
        "severity_f1": round(f1_score(y_true_severity, y_pred_severity, average="macro"), 3),

        "counter_type_acc": round(accuracy_score(y_true_counter_type, y_pred_counter_type), 3),
        "counter_type_f1": round(f1_score(y_true_counter_type, y_pred_counter_type, average="macro"), 3),

        # Multilabel (micro average)
        "identity_acc": round(accuracy_score(np.array(y_true_id).flatten(),
                                             np.array(y_pred_id).flatten()), 3),
        "identity_f1": round(f1_score(y_true_id, y_pred_id, average="micro"), 3)
    }

    return metrics

# ==============================================================
# Run experiments (10 iterations per batch)
# ==============================================================

for batch_size in batch_sizes:
    print(f"\n=== Testing batch size {batch_size} for {num_iterations} iterations ===")
    all_iter_results = []
    all_iter_times = []

    for iteration in range(num_iterations):
        print(f"Iteration {iteration + 1}/{num_iterations}")
        batch_times, preds = [], {}

        for i in range(0, len(examples), batch_size):
            batch = examples[i:i + batch_size]
            batched_messages = build_messages(batch)
            start = time.time()
            resp = model.query(messages=batched_messages)
            elapsed = time.time() - start
            batch_times.append(elapsed)

            raw_text = resp.choices[0].message.content
            parsed = parse_model_output(raw_text)
            for item in parsed:
                if "id" in item:
                    preds[int(item["id"])] = item

            # ✅ Pydantic v2 compatible model stats logging
            with open("usage_log.json", "w") as f:
                json.dump(model.stats.model_dump(), f, indent=2)

        # --- Record metrics for this iteration ---
        metrics = compute_metrics(preds, ground_truth)
        total_time, avg_time = sum(batch_times), mean(batch_times)
        all_iter_times.append(avg_time)
        all_iter_results.append(metrics)

    # --- Aggregate over 10 iterations ---
    avg_metrics = {
        "hate_acc": round(mean([m["hate_acc"] for m in all_iter_results]), 3),
        "hate_f1": round(mean([m["hate_f1"] for m in all_iter_results]), 3),
        "counter_acc": round(mean([m["counter_acc"] for m in all_iter_results]), 3),
        "counter_f1": round(mean([m["counter_f1"] for m in all_iter_results]), 3),
    }

    results_summary.append({
        "batch_size": batch_size,
        "iterations": num_iterations,
        "avg_time_s": round(mean(all_iter_times), 2),
        **avg_metrics,
    })

# ==============================================================
# Final Summary
# ==============================================================

print("\n=== Batch Size Evaluation Summary (Averaged over 10 iterations) ===")
for r in results_summary:
    print(r)



=== Testing batch size 50 for 1 iterations ===
Iteration 1/1

=== Batch Size Evaluation Summary (Averaged over 10 iterations) ===
{'batch_size': 50, 'iterations': 1, 'avg_time_s': 124.22, 'hate_acc': 0.9, 'hate_f1': 0.783, 'counter_acc': 0.74, 'counter_f1': 0.133}


In [10]:
import time, json

# --- Run one test with batch size 50 ---
batch_size = 50
preds = {}
all_outputs = []  # raw model responses (for debugging)

print(f"\n=== Running single batch test with batch size {batch_size} ===")

batch_times = []
for i in range(0, len(examples), batch_size):
    batch = examples[i:i + batch_size]
    batched_messages = build_messages(batch)

    start = time.time()
    resp = model.query(messages=batched_messages)
    elapsed = time.time() - start
    batch_times.append(elapsed)

    raw_text = resp.choices[0].message.content
    all_outputs.append({
        "batch_index": i // batch_size,
        "time_s": round(elapsed, 2),
        "raw_output": raw_text
    })

    parsed = parse_model_output(raw_text)
    for item in parsed:
        if "id" in item:
            preds[int(item["id"])] = item

print(f"✅ Completed batch of size {batch_size} in {round(sum(batch_times), 2)}s")

# --- Save outputs ---
with open("model_raw_outputs_batch50.json", "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, indent=2, ensure_ascii=False)

with open("model_parsed_preds_batch50.json", "w", encoding="utf-8") as f:
    json.dump(preds, f, indent=2, ensure_ascii=False)

print("💾 Saved:")
print("  - model_raw_outputs_batch50.json (raw model responses)")
print("  - model_parsed_preds_batch50.json (parsed per-post predictions)")



=== Running single batch test with batch size 50 ===
✅ Completed batch of size 50 in 130.15s
💾 Saved:
  - model_raw_outputs_batch50.json (raw model responses)
  - model_parsed_preds_batch50.json (parsed per-post predictions)


In [11]:
import time, json

# --- Load annotated ground truth ---
with open("50_annotated_examples.json", "r", encoding="utf-8") as f:
    ground_truth = {int(d["id"]): d for d in json.load(f)}

# --- Prepare data (using your parser + few-shot messages setup) ---
examples = list(parser.iter_items())
batch_size = 50

# --- Helper functions ---
def build_messages(batch):
    """Helper to construct per-batch messages for model query."""
    return [
        messages[0],         # system prompt
        *messages[1:-1],     # few-shot examples
        {
            "role": "user",
            "content": "\n\n".join(
                [f"ID: {ex['id']}\nText: {ex['text']}\nParent_Text: {ex['parent_text']}" for ex in batch]
            ),
        },
    ]

def parse_model_output(raw_text):
    """Safely parse model JSON output into Python list."""
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        try:
            start = raw_text.find("[")
            end = raw_text.rfind("]") + 1
            return json.loads(raw_text[start:end])
        except Exception:
            return []

# --- Run single batch test ---
preds = {}
all_outputs = []  # stores raw responses and timings
batch_times = []

print(f"\n=== Running single batch test with batch size {batch_size} ===")

for i in range(0, len(examples), batch_size):
    batch = examples[i:i + batch_size]
    batched_messages = build_messages(batch)

    start = time.time()
    resp = model.query(messages=batched_messages)
    elapsed = time.time() - start
    batch_times.append(elapsed)

    raw_text = resp.choices[0].message.content
    all_outputs.append({
        "batch_index": i // batch_size,
        "time_s": round(elapsed, 2),
        "raw_output": raw_text
    })

    parsed = parse_model_output(raw_text)
    for item in parsed:
        if "id" in item:
            preds[int(item["id"])] = item

print(f"✅ Completed batch of size {batch_size} in {round(sum(batch_times), 2)}s")

# --- Save results ---
with open("model_raw_outputs_batch50.json", "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, indent=2, ensure_ascii=False)

with open("model_parsed_preds_batch50.json", "w", encoding="utf-8") as f:
    json.dump(preds, f, indent=2, ensure_ascii=False)

print("💾 Saved:")
print("  - model_raw_outputs_batch50.json (raw GPT responses)")
print("  - model_parsed_preds_batch50.json (parsed predictions per post)")



=== Running single batch test with batch size 50 ===
✅ Completed batch of size 50 in 173.16s
💾 Saved:
  - model_raw_outputs_batch50.json (raw GPT responses)
  - model_parsed_preds_batch50.json (parsed predictions per post)


In [12]:
'''
# === Batch Size + Ground Truth Evaluation ===
import time, json, numpy as np
from statistics import mean
from sklearn.metrics import accuracy_score, f1_score

# --- Load ground truth labels ---
with open("50_annotated_examples.json", "r", encoding="utf-8") as f:
    ground_truth = {int(d["id"]): d for d in json.load(f)}

# --- Prepare data ---
examples = list(parser.iter_items())
batch_sizes = [32, 50]
results_summary = []

def build_messages(batch):
    """Helper to construct per-batch messages for model query."""
    return [
        messages[0],         # system prompt
        *messages[1:-1],     # few-shot examples
        {
            "role": "user",
            "content": "\n\n".join(
                [f"ID: {ex['id']}\nText: {ex['text']}\nParent_Text: {ex['parent_text']}" for ex in batch]
            ),
        },
    ]

def parse_model_output(raw_text):
    """Safely parse model JSON output into Python list."""
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        # Try to recover from leading/trailing text
        try:
            start = raw_text.find("[")
            end = raw_text.rfind("]") + 1
            return json.loads(raw_text[start:end])
        except Exception:
            return []

def compute_metrics(preds, gold):
    """Compute accuracy & F1 for hate_speech and counterspeech."""
    y_true_hate, y_pred_hate = [], []
    y_true_counter, y_pred_counter = [], []
    for pid, p in preds.items():
        g = gold.get(pid)
        if not g:
            continue
        y_true_hate.append(g["hate_speech"])
        y_pred_hate.append(p.get("hate_speech", 0))
        y_true_counter.append(g["counterspeech"])
        y_pred_counter.append(p.get("counterspeech", 0))
    return {
        "hate_acc": round(accuracy_score(y_true_hate, y_pred_hate), 3),
        "hate_f1": round(f1_score(y_true_hate, y_pred_hate), 3),
        "counter_acc": round(accuracy_score(y_true_counter, y_pred_counter), 3),
        "counter_f1": round(f1_score(y_true_counter, y_pred_counter), 3),
    }

# --- Run experiments ---
for batch_size in batch_sizes:
    print(f"\n=== Testing batch size {batch_size} ===")
    batch_times, preds = [], {}

    for i in range(0, len(examples), batch_size):
        batch = examples[i:i + batch_size]
        batched_messages = build_messages(batch)
        start = time.time()
        resp = model.query(messages=batched_messages)
        elapsed = time.time() - start
        batch_times.append(elapsed)

        raw_text = resp.choices[0].message.content
        parsed = parse_model_output(raw_text)
        for item in parsed:
            if "id" in item:
                preds[int(item["id"])] = item

    with open("usage_log.json", "w") as f:
        json.dump(model.stats.dict(), f, indent=2)

    metrics = compute_metrics(preds, ground_truth)
    total_time, avg_time = sum(batch_times), mean(batch_times)
    results_summary.append({
        "batch_size": batch_size,
        "batches": len(batch_times),
        "total_time_s": round(total_time, 2),
        "avg_time_s": round(avg_time, 2),
        **metrics,
    })

# --- Display summary ---
print("\n=== Batch Size Evaluation Summary ===")
for r in results_summary:
    print(r)
'''

'\n# === Batch Size + Ground Truth Evaluation ===\nimport time, json, numpy as np\nfrom statistics import mean\nfrom sklearn.metrics import accuracy_score, f1_score\n\n# --- Load ground truth labels ---\nwith open("50_annotated_examples.json", "r", encoding="utf-8") as f:\n    ground_truth = {int(d["id"]): d for d in json.load(f)}\n\n# --- Prepare data ---\nexamples = list(parser.iter_items())\nbatch_sizes = [32, 50]\nresults_summary = []\n\ndef build_messages(batch):\n    """Helper to construct per-batch messages for model query."""\n    return [\n        messages[0],         # system prompt\n        *messages[1:-1],     # few-shot examples\n        {\n            "role": "user",\n            "content": "\n\n".join(\n                [f"ID: {ex[\'id\']}\nText: {ex[\'text\']}\nParent_Text: {ex[\'parent_text\']}" for ex in batch]\n            ),\n        },\n    ]\n\ndef parse_model_output(raw_text):\n    """Safely parse model JSON output into Python list."""\n    try:\n        return js

In [13]:
# Print the full raw response object (optional, for debugging)
print(completion)

# Print just the text output
print("\n=== MODEL RESPONSE ===\n")
print(completion.choices[0].message["content"])

# If you want *all* responses (sometimes n > 1):
for i, choice in enumerate(completion.choices, start=1):
    print(f"\n=== RESPONSE {i} ===\n")
    print(choice.message["content"])

NameError: name 'completion' is not defined

# Usage with vLLM
- Install vLLM and launch the server. It will give you a host url.
- Example command: `CUDA_VISIBLE_DEVICES=0 vllm serve Qwen/Qwen3-1.7B --trust-remote-code`
- Running this command should give you a host url. You can use it as follows.

In [ ]:
# In your script, import the APIModelConfig and LiteLLMModel classes
config = APIModelConfig(model_name="hosted_vllm/Qwen3-1.7B", temperature=0.7, top_p=1.0, max_tokens=1000, host_url="http://0.0.0.0:8000")
model = LiteLLMModel(config=config, name="hosted_vllm/Qwen3-1.7B", logger=logging.getLogger(__name__))

# Prepare the messages
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of the moon?"},
]

# Call the model
response = model.query(messages=messages)
print(response.choices[0].message.content)


Using custom host URL: http://0.0.0.0:8000. Cost management will not be available. Register the model with litellm to enable cost management. See https://docs.litellm.ai/docs/completion/token_usage#9-register_model
Error querying hosted_vllm/Qwen3-1.7B with tools
Traceback (most recent call last):
  File "/data4/anika/UCSBCounterspeechProject/.venv/lib/python3.11/site-packages/litellm/llms/openai/openai.py", line 736, in completion
    raise e
  File "/data4/anika/UCSBCounterspeechProject/.venv/lib/python3.11/site-packages/litellm/llms/openai/openai.py", line 664, in completion
    ) = self.make_sync_openai_chat_completion_request(
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data4/anika/UCSBCounterspeechProject/.venv/lib/python3.11/site-packages/litellm/litellm_core_utils/logging_utils.py", line 237, in sync_wrapper
    result = func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/data4/anika/UCSBCounterspeechProject/.venv/lib/python3.11/site-packages/l


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



NotFoundError: litellm.NotFoundError: NotFoundError: Hosted_vllmException - Error code: 404 - {'detail': 'Not Found'}